# 02 — Fine-Tuning

Tokenize the processed CUAD dataset and fine-tune a classifier.

**Note on CUAD size:** The full dataset has ~13k positive + many negative examples.  
For fast iteration, `SAMPLE_N` below lets you train on a subset first.

| Preset | Model | Hardware | Time (full) |
|--------|-------|----------|-------------|
| `distilbert` | distilbert-base-uncased | CPU ok | ~1h |
| `deberta` | microsoft/deberta-v3-small | 4 GB RAM | ~2h |
| `mistral` | mistralai/Mistral-7B-v0.1 | GPU + 8 GB VRAM | ~4h+ |

> **Kernel:** `Legal AI`

In [1]:
import json, sys
from pathlib import Path

import numpy as np
import pandas as pd
import yaml
from datasets import Dataset, DatasetDict
from sklearn.model_selection import train_test_split
from transformers import AutoTokenizer

ROOT = Path().resolve().parent
sys.path.insert(0, str(ROOT))

with open(ROOT / 'config.yaml') as f:
    CFG = yaml.safe_load(f)

PROCESSED_DIR = ROOT / CFG['dataset']['processed_dir']
OUTPUT_DIR    = ROOT / CFG['training']['output_dir']
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

PRESET_NAME = CFG['model']['preset']
PRESET      = CFG['model']['presets'][PRESET_NAME]
BASE_MODEL  = PRESET['base_model']
MAX_LEN     = CFG['dataset']['max_seq_length']
SEED        = CFG['training']['seed']

# ── Optional: limit dataset size for fast prototyping ─────────────────────────
# Set to None to use the full dataset
SAMPLE_N = None   # e.g. 5000 for a quick test run

# ── Optional: exclude negative examples ('No X') for binary classification ───
INCLUDE_NEGATIVES = True

print(f'Preset : {PRESET_NAME}  ({BASE_MODEL})')
print(f'Max len: {MAX_LEN} tokens')
print(f'Sample : {SAMPLE_N or "full dataset"}')

Preset : distilbert  (distilbert-base-uncased)
Max len: 512 tokens
Sample : full dataset


## 1. Load processed data

In [2]:
csv_path = PROCESSED_DIR / 'contracts_clean.csv'
if not csv_path.exists():
    raise FileNotFoundError('Run 01_prepare_data.ipynb first.')

df = pd.read_csv(csv_path)

with open(PROCESSED_DIR / 'label2id.json') as f:
    label2id = json.load(f)
with open(PROCESSED_DIR / 'id2label.json') as f:
    id2label = {int(k): v for k, v in json.load(f).items()}

print(f'Loaded: {len(df):,} rows, {len(label2id)} labels')
print(df['label'].value_counts().head(10).to_string())

Loaded: 20,744 rows, 52 labels
label
Parties             2287
Governing Law       1334
Other               1144
Audit Rights        1061
Anti-Assignment      971
License Grant        903
Insurance            880
Document Name        861
Expiration Date      794
Cap On Liability     774


In [3]:
# Filter negatives if not wanted
if not INCLUDE_NEGATIVES:
    before = len(df)
    df = df[~df['label'].str.startswith('No ')].copy()
    # Re-build label map without negative classes
    label_names = sorted(df['label'].unique())
    label2id    = {lbl: i for i, lbl in enumerate(label_names)}
    id2label    = {i: lbl for lbl, i in label2id.items()}
    df['label_id'] = df['label'].map(label2id)
    print(f'Dropped negatives: {before:,} → {len(df):,} rows, {len(label2id)} labels')

# Optional subsampling (stratified)
if SAMPLE_N and SAMPLE_N < len(df):
    df = df.groupby('label', group_keys=False).apply(
        lambda g: g.sample(min(len(g), max(1, int(SAMPLE_N * len(g) / len(df)))),
                           random_state=SEED)
    ).reset_index(drop=True)
    print(f'Sampled to {len(df):,} rows')

NUM_LABELS = len(label2id)
print(f'\nFinal: {len(df):,} samples, {NUM_LABELS} labels')


Final: 20,744 samples, 52 labels


## 2. Train / Val / Test split

In [4]:
# ── Robust stratified split ───────────────────────────────────────────────────
# We need each class to have >=4 samples so that after the 80/20 first split
# the temp_df (20%) still has >=2 samples per class for the second split.
# Classes below that threshold are merged into an 'Other' bucket instead of
# being dropped entirely (preserves data, avoids silent data loss).

MIN_SAMPLES = 4   # minimum per class across the whole df before splitting

counts = df['label_id'].value_counts()
rare   = counts[counts < MIN_SAMPLES].index.tolist()

if rare:
    rare_labels = df[df['label_id'].isin(rare)]['label'].unique().tolist()
    print(f'Merging {len(rare)} rare classes into "Other": {rare_labels}')
    df.loc[df['label_id'].isin(rare), 'label'] = 'Other'

# Rebuild a clean contiguous label map after any merges
label_names = sorted(df['label'].unique())
label2id    = {lbl: i for i, lbl in enumerate(label_names)}
id2label    = {i: lbl for lbl, i in label2id.items()}
df['label_id'] = df['label'].map(label2id)
NUM_LABELS = len(label2id)
print(f'Labels after merge: {NUM_LABELS}')

# First split: 80% train, 20% temp
train_df, temp_df = train_test_split(
    df, test_size=1 - CFG['dataset']['train_split'],
    stratify=df['label_id'], random_state=SEED
)

# Second split: temp → 50% val, 50% test
# Guard again: if any class in temp_df has only 1 sample, merge to Other
temp_counts  = temp_df['label_id'].value_counts()
rare_in_temp = temp_counts[temp_counts < 2].index.tolist()
if rare_in_temp:
    rare_lbls = temp_df[temp_df['label_id'].isin(rare_in_temp)]['label'].unique().tolist()
    print(f'Merging {len(rare_lbls)} rare-in-temp classes into "Other": {rare_lbls}')
    temp_df = temp_df.copy()
    temp_df.loc[temp_df['label_id'].isin(rare_in_temp), 'label'] = 'Other'
    temp_df['label_id'] = temp_df['label'].map(label2id).fillna(label2id.get('Other', 0)).astype(int)

val_df, test_df = train_test_split(
    temp_df, test_size=0.5, stratify=temp_df['label_id'], random_state=SEED
)

print(f'Train : {len(train_df):,}')
print(f'Val   : {len(val_df):,}')
print(f'Test  : {len(test_df):,}')

Merging 4 rare classes into "Other": ['No No-Solicit Of Customers', 'No Exclusivity', 'No Parties', 'No Governing Law']
Labels after merge: 48
Merging 1 rare-in-temp classes into "Other": ['No Non-Compete']
Train : 16,595
Val   : 2,074
Test  : 2,075


## 3. Tokenize

In [5]:
print(f'Loading tokenizer: {BASE_MODEL} ...')
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)

# For Mistral-style models that have no pad token
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

def to_hf(split_df):
    return Dataset.from_dict({
        'text'  : split_df['text'].tolist(),
        'labels': split_df['label_id'].tolist(),
    })

raw_ds = DatasetDict({
    'train'     : to_hf(train_df),
    'validation': to_hf(val_df),
    'test'      : to_hf(test_df),
})

def tokenize(batch):
    return tokenizer(
        batch['text'],
        truncation=True, padding='max_length', max_length=MAX_LEN,
    )

tokenized_ds = raw_ds.map(
    tokenize, batched=True, remove_columns=['text'], desc='Tokenizing'
)
print(tokenized_ds)

Loading tokenizer: distilbert-base-uncased ...


Tokenizing:   0%|          | 0/16595 [00:00<?, ? examples/s]

Tokenizing:   0%|          | 0/2074 [00:00<?, ? examples/s]

Tokenizing:   0%|          | 0/2075 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['labels', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 16595
    })
    validation: Dataset({
        features: ['labels', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 2074
    })
    test: Dataset({
        features: ['labels', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 2075
    })
})


## 4. Load model

In [6]:
USE_LORA = PRESET.get('use_lora', False)

if USE_LORA:
    import torch
    from peft import LoraConfig, TaskType, get_peft_model
    from transformers import AutoModelForSequenceClassification, BitsAndBytesConfig

    bnb = BitsAndBytesConfig(
        load_in_4bit=PRESET.get('load_in_4bit', True),
        bnb_4bit_quant_type='nf4',
        bnb_4bit_compute_dtype=torch.float16,
    )
    model = AutoModelForSequenceClassification.from_pretrained(
        BASE_MODEL, num_labels=NUM_LABELS,
        quantization_config=bnb, device_map='auto',
        id2label=id2label, label2id=label2id,
    )
    lora_cfg = LoraConfig(
        task_type=TaskType.SEQ_CLS,
        r=PRESET.get('lora_r', 16), lora_alpha=PRESET.get('lora_alpha', 32),
        lora_dropout=PRESET.get('lora_dropout', 0.05),
        target_modules=PRESET.get('lora_target_modules', ['q_proj', 'v_proj']),
    )
    model = get_peft_model(model, lora_cfg)
    model.print_trainable_parameters()
else:
    from transformers import AutoModelForSequenceClassification
    model = AutoModelForSequenceClassification.from_pretrained(
        BASE_MODEL, num_labels=NUM_LABELS,
        id2label=id2label, label2id=label2id,
        ignore_mismatched_sizes=True,
    )

total     = sum(p.numel() for p in model.parameters())
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Parameters: {total/1e6:.1f}M total  |  {trainable/1e6:.1f}M trainable')

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
pre_classifier.weight   | MISSING    | 
pre_classifier.bias     | MISSING    | 
classifier.bias         | MISSING    | 
classifier.weight       | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Parameters: 67.0M total  |  67.0M trainable


## 5. Train

In [7]:
import torch
import evaluate as hf_evaluate
from transformers import DataCollatorWithPadding, Trainer, TrainingArguments

# ── GPU check ────────────────────────────────────────────────────────────────
if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    vram_gb  = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f'✅ GPU : {gpu_name}  ({vram_gb:.1f} GB VRAM)')
    print(f'   CUDA: {torch.version.cuda}')
else:
    print('⚠️  No CUDA GPU detected — training will use CPU (slow).')
    print('   Make sure you installed torch with CUDA support:')
    print('   uv pip install torch --index-url https://download.pytorch.org/whl/cu121')

# ── metrics ──────────────────────────────────────────────────────────────────
acc_metric = hf_evaluate.load('accuracy')
f1_metric  = hf_evaluate.load('f1')

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return {
        'accuracy': acc_metric.compute(predictions=preds, references=labels)['accuracy'],
        'f1'      : f1_metric.compute(predictions=preds, references=labels, average='macro')['f1'],
    }

# ── training args ─────────────────────────────────────────────────────────────
tcfg = CFG['training']
training_args = TrainingArguments(
    output_dir                  = str(OUTPUT_DIR),
    num_train_epochs            = tcfg['num_epochs'],
    per_device_train_batch_size = tcfg['batch_size'],
    per_device_eval_batch_size  = tcfg['batch_size'] * 2,
    gradient_accumulation_steps = tcfg.get('gradient_accumulation_steps', 1),
    learning_rate               = tcfg['learning_rate'],
    warmup_ratio                = tcfg['warmup_ratio'],
    weight_decay                = tcfg['weight_decay'],
    eval_strategy               = tcfg['evaluation_strategy'],
    save_strategy               = tcfg['save_strategy'],
    load_best_model_at_end      = tcfg['load_best_model_at_end'],
    metric_for_best_model       = tcfg['metric_for_best_model'],
    fp16                        = tcfg['fp16'] and torch.cuda.is_available(),
    dataloader_pin_memory       = tcfg.get('dataloader_pin_memory', False) and torch.cuda.is_available(),
    logging_steps               = tcfg['logging_steps'],
    seed                        = tcfg['seed'],
    report_to                   = 'none',
)

eff_batch = tcfg['batch_size'] * tcfg.get('gradient_accumulation_steps', 1)
print(f'Effective batch size : {eff_batch}  ({tcfg["batch_size"]} × {tcfg.get("gradient_accumulation_steps",1)} grad accum)')
print(f'fp16                 : {training_args.fp16}')
print(f'Device               : {training_args.device}')

# ── trainer ───────────────────────────────────────────────────────────────────
trainer = Trainer(
    model            = model,
    args             = training_args,
    train_dataset    = tokenized_ds['train'],
    eval_dataset     = tokenized_ds['validation'],
    processing_class = tokenizer,
    data_collator    = DataCollatorWithPadding(tokenizer),
    compute_metrics  = compute_metrics,
)

print('\nStarting training...')
train_result = trainer.train()
print(train_result.metrics)


✅ GPU : NVIDIA GeForce RTX 3060 Laptop GPU  (6.4 GB VRAM)
   CUDA: 12.6


[transformers] warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


Effective batch size : 16  (8 × 2 grad accum)
fp16                 : True
Device               : cuda:0

Starting training...


Epoch,Training Loss,Validation Loss,Accuracy,F1
1,1.911871,0.799299,0.815333,0.506622
2,0.986266,0.478741,0.868852,0.598921
3,0.712404,0.398569,0.883799,0.666576
4,0.446053,0.391678,0.891514,0.698616
5,0.335478,0.387138,0.891032,0.711294


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

{'train_runtime': 1213.2818, 'train_samples_per_second': 68.389, 'train_steps_per_second': 4.278, 'total_flos': 1.10004992077824e+16, 'train_loss': 1.3822815556976378, 'epoch': 5.0}


## 6. Save model + tokenizer + dataset for evaluation

In [8]:
BEST_MODEL_DIR = OUTPUT_DIR / 'best_model'
trainer.save_model(str(BEST_MODEL_DIR))
tokenizer.save_pretrained(str(BEST_MODEL_DIR))

with open(OUTPUT_DIR / 'train_metrics.json', 'w') as f:
    json.dump(train_result.metrics, f, indent=2)

# Save tokenized test set for notebook 03
DATASET_DIR = PROCESSED_DIR / 'hf_dataset'
tokenized_ds.save_to_disk(str(DATASET_DIR))
with open(DATASET_DIR / 'label2id.json', 'w') as f: json.dump(label2id, f)
with open(DATASET_DIR / 'id2label.json', 'w') as f: json.dump(id2label, f)

print(f'Model saved  → {BEST_MODEL_DIR}')
print(f'Dataset saved → {DATASET_DIR}')

# Quick test eval
test_results = trainer.evaluate(tokenized_ds['test'])
print('\nTest results:')
for k, v in test_results.items():
    print(f'  {k}: {v:.4f}' if isinstance(v, float) else f'  {k}: {v}')

print('\n✅ Ready for notebook 03_evaluate.ipynb')

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Saving the dataset (0/1 shards):   0%|          | 0/16595 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/2074 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/2075 [00:00<?, ? examples/s]

Model saved  → models\finetuned\best_model
Dataset saved → data\processed\hf_dataset


Training Loss,Validation Loss,Epoch,Accuracy,F1
0.335478,0.370097,5,0.898313,0.723306



Test results:
  eval_loss: 0.3701
  eval_accuracy: 0.8983
  eval_f1: 0.7233

✅ Ready for notebook 03_evaluate.ipynb
